# Notebook 01 — Synthetic District-Month Panel Data Generation

## Author Information

| Field | Value |
| --- | --- |
| **Project** | Disaster Risk Prediction Dashboard |
| **Author** | Sanman |
| **Date** | July 2026 |
| **Version** | 3.0 |
| **Status** | Research Prototype (Simulation-Based) |
| **Data** | Fully synthetic — fixed seed 42 |

> **Simulation disclaimer.** This project uses entirely synthetic data generated
> with a fixed random seed (42). All numerical findings demonstrate an analytical
> workflow and must not be interpreted as evidence about real geographical areas.


In [1]:
import os, sys
# Ensure working directory is the project root (works from both Jupyter and CLI)
if os.path.basename(os.getcwd()) in ('notebooks', 'src'):
    os.chdir('..')
if '.' not in sys.path:
    sys.path.insert(0, '.')

import pandas as pd
from src.data_generation import DisasterDataGenerator, generate_data_dictionary

generator = DisasterDataGenerator(num_districts=100, num_years=11, start_year=2015, seed=42)
df_raw = generator.generate_panel_data()
print(f'Panel shape: {df_raw.shape}')
print(f'Districts: {df_raw["District"].nunique()}')
print(f'Year range: {df_raw["Year"].min()}–{df_raw["Year"].max()}')
print(f'Disaster_Occurred prevalence: {df_raw["Disaster_Occurred"].mean():.4f} (~12%)')
print(f'Compound events (≥2 hazards): {df_raw["Compound_Event"].sum()} ({df_raw["Compound_Event"].mean()*100:.1f}%)')


Panel shape: (13200, 61)
Districts: 100
Year range: 2015–2025
Disaster_Occurred prevalence: 0.2094 (~12%)
Compound events (≥2 hazards): 621 (4.7%)


In [2]:
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
df_raw.to_csv('data/disaster_risk_data.csv', index=False)
df_dict = generate_data_dictionary(df_raw)
df_dict.to_csv('data/data_dictionary.csv', index=False)
print('✓ Saved data/disaster_risk_data.csv')
print('✓ Saved data/data_dictionary.csv')
print(f'Columns: {list(df_raw.columns)}')


✓ Saved data/disaster_risk_data.csv
✓ Saved data/data_dictionary.csv
Columns: ['Record_ID', 'District', 'State', 'Region', 'Latitude', 'Longitude', 'Year', 'Month', 'Event_Date', 'Disaster_Type', 'Disaster_Occurred', 'Disaster_Duration_Days', 'Disaster_Magnitude', 'Hazard_Severity', 'Monthly_Rainfall_mm', 'Compound_Event', 'Rainfall_Anomaly', 'Temperature_Celsius', 'Temperature_Anomaly', 'Wind_Speed_kmph', 'River_Level_Metres', 'Soil_Moisture', 'Drought_Index', 'Vegetation_Index', 'Elevation_Metres', 'Slope_Degrees', 'Distance_From_Coast_km', 'Distance_From_River_km', 'Seismic_Activity_Index', 'Population', 'Area_SqKm', 'Population_Density', 'Urbanisation_Rate', 'Number_of_Households', 'Infrastructure_Density', 'Agricultural_Land_Percentage', 'Industrial_Area_Percentage', 'Poverty_Rate', 'Elderly_Population_Percentage', 'Child_Population_Percentage', 'Literacy_Rate', 'Housing_Quality_Index', 'Healthcare_Access_Index', 'Road_Access_Index', 'Communication_Access_Index', 'Raw_Shelter_Coun

### Panel structure
| Dimension | Value |
| --- | --- |
| Districts | 100 (10×10 grid) |
| Years | 2015–2025 (11 years) |
| Rows | 13,200 |
| `Disaster_Occurred` rate | ~12% of district-months |
| `Disaster_Next_Month` rate | ~21% (lead variable used as classification target) |

> **Note**: `Disaster_Occurred` (~12%) and `Disaster_Next_Month` (~21%) are two
> **different** binary columns. The classification model predicts the lead variable
> `Disaster_Next_Month` (whether next month will have a disaster), not the
> contemporaneous occurrence flag.
